# SAM2-UNet v2 — +Dice Loss + Boundary Loss

Ablation step 2: thêm Dice Loss và Boundary Loss vào loss function.

**Thay đổi:** chỉ sửa loss, kiến trúc giữ nguyên 100%.

## 1. Setup

In [ ]:
!git clone https://github.com/WZH0120/SAM2-UNet.git

Cloning into 'SAM2-UNet'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 316 (delta 85), reused 58 (delta 58), pack-reused 208 (from 2)
Receiving objects: 100% (316/316), 3.26 MiB | 8.29 MiB/s, done.
Resolving deltas: 100% (125/125), done.


In [ ]:
%cd SAM2-UNet

/content/SAM2-UNet


In [ ]:
!pip install -r requirements.txt

Looking in links: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 6.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
!pip install -q gdown

## 2. Download checkpoint

In [ ]:
!mkdir -p /content/checkpoints
!wget -O /content/checkpoints/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

--2026-05-14 16:11:07--  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.4, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 897952466 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘/content/checkpoints/sam2_hiera_large.pt’

/content/checkpoint 100%[===================>] 856.35M   363MB/s    in 2.4s    

2026-05-14 16:11:09 (363 MB/s) - ‘/content/checkpoints/sam2_hiera_large.pt’ saved [897952466/897952466]



In [ ]:
!ls -lh /content/checkpoints

total 857M
-rw-r--r-- 1 root root 857M Jul 28  2024 sam2_hiera_large.pt


## 3. Download dataset

> Bỏ qua nếu data đã có từ lần chạy baseline.

In [ ]:
!mkdir -p /content/data/Polyp
%cd /content/data/Polyp

# TrainDataset: Kvasir-SEG + CVC-ClinicDB
!gdown --fuzzy "https://drive.google.com/file/d/1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb/view?usp=sharing" -O TrainDataset.zip

# TestDataset: Kvasir, CVC-ClinicDB, CVC-ColonDB, CVC-300, ETIS
!gdown --fuzzy "https://drive.google.com/file/d/1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao/view?usp=sharing" -O TestDataset.zip

!unzip -q TrainDataset.zip
!unzip -q TestDataset.zip

/content/data/Polyp
Downloading...
From (original): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb
From (redirected): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb&confirm=t&uuid=ed49c617-fe35-4179-875e-edb6e2d25574
To: /content/data/Polyp/TrainDataset.zip
100% 419M/419M [00:05<00:00, 70.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao
From (redirected): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao&confirm=t&uuid=e874c45f-ff7e-4563-a7ec-ab1465366106
To: /content/data/Polyp/TestDataset.zip
100% 343M/343M [00:02<00:00, 132MB/s]


In [ ]:
import os
from pathlib import Path

root = Path("/content/data/Polyp")
check_paths = [
    "TrainDataset/images", "TrainDataset/masks",
    "TestDataset/Kvasir/images", "TestDataset/Kvasir/masks",
    "TestDataset/CVC-ClinicDB/images", "TestDataset/CVC-ClinicDB/masks",
    "TestDataset/CVC-ColonDB/images", "TestDataset/CVC-ColonDB/masks",
    "TestDataset/CVC-300/images", "TestDataset/CVC-300/masks",
    "TestDataset/ETIS-LaribPolypDB/images", "TestDataset/ETIS-LaribPolypDB/masks",
]
for p in check_paths:
    full = root / p
    print(p, "=>", len(os.listdir(full)) if full.exists() else "MISSING")

TrainDataset/images => MISSING
TrainDataset/masks => 1450
TestDataset/Kvasir/images => 100
TestDataset/Kvasir/masks => 100
TestDataset/CVC-ClinicDB/images => 62
TestDataset/CVC-ClinicDB/masks => 62
TestDataset/CVC-ColonDB/images => 380
TestDataset/CVC-ColonDB/masks => 380
TestDataset/CVC-300/images => 60
TestDataset/CVC-300/masks => 60
TestDataset/ETIS-LaribPolypDB/images => 196
TestDataset/ETIS-LaribPolypDB/masks => 196


In [ ]:
# Fix symlink nếu cần (từ notebook baseline)
import os
src = "/content/data/Polyp/TrainDataset/image"
dst = "/content/data/Polyp/TrainDataset/images"
if os.path.exists(src) and not os.path.exists(dst):
    os.symlink(src, dst)
    print("Symlink created")
else:
    print("OK - no symlink needed")

Symlink created


## 4. Smoke test (1 epoch) — kiểm tra loss chạy được

In [ ]:
%cd /content/SAM2-UNet

# Write train_v2.py
train_v2_src = open("/content/train_v2.py").read() if os.path.exists("/content/train_v2.py") else None
print("train_v2.py ready" if train_v2_src else "Will write below")

/content/SAM2-UNet
train_v2.py ready


In [ ]:
%%writefile /content/SAM2-UNet/train_v2.py
import os
import argparse
import torch
import torch.optim as opt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from dataset import FullDataset
from SAM2UNet import SAM2UNet

# ── Loss functions ────────────────────────────────────────────────────────────

def structure_loss(pred, mask):
    """Original wBCE + wIoU — giữ nguyên từ paper gốc."""
    weit = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    )
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduce='none')
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))
    pred_sig = torch.sigmoid(pred)
    inter = ((pred_sig * mask) * weit).sum(dim=(2, 3))
    union = ((pred_sig + mask) * weit).sum(dim=(2, 3))
    wiou  = 1 - (inter + 1) / (union - inter + 1)
    return (wbce + wiou).mean()

def dice_loss(pred, mask, smooth=1.0):
    """Dice loss — bù đắp điểm yếu của wIoU trên polyp nhỏ."""
    pred_sig  = torch.sigmoid(pred)
    pred_flat = pred_sig.view(pred_sig.size(0), -1)
    mask_flat = mask.view(mask.size(0), -1)
    intersection = (pred_flat * mask_flat).sum(dim=1)
    dice = (2.0 * intersection + smooth) / (
        pred_flat.sum(dim=1) + mask_flat.sum(dim=1) + smooth
    )
    return (1 - dice).mean()

def get_boundary(mask, kernel_size=5):
    """Extract boundary GT bằng morphological erosion: boundary = mask - erode(mask)."""
    pad    = kernel_size // 2
    eroded = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=kernel_size, stride=1, padding=pad)
    return torch.clamp(mask - eroded, min=0.0)

def boundary_loss(pred, mask, kernel_size=5):
    """BCE trên vùng boundary của GT mask — supervise trực tiếp vùng khó nhất."""
    boundary_gt = get_boundary(mask, kernel_size)
    bce    = F.binary_cross_entropy_with_logits(pred, boundary_gt, reduce='none')
    weight = (boundary_gt > 0).float()
    num_px = weight.sum().clamp(min=1.0)
    return (bce * weight).sum() / num_px

def combined_loss(pred, mask, dice_w=0.5, boundary_w=0.3):
    return structure_loss(pred, mask) + dice_w * dice_loss(pred, mask) + boundary_w * boundary_loss(pred, mask)

# ── Config ────────────────────────────────────────────────────────────────────

HIERA_PATH       = "/content/checkpoints/sam2_hiera_large.pt"
TRAIN_IMAGE_PATH = "/content/data/Polyp/TrainDataset/images/"
TRAIN_MASK_PATH  = "/content/data/Polyp/TrainDataset/masks/"
SAVE_PATH        = "/content/ckpt/sam2unet_v2_dice_boundary/"
EPOCH            = 20
BATCH_SIZE       = 12
LR               = 0.001
WEIGHT_DECAY     = 5e-4
DICE_W           = 0.3
BOUNDARY_W       = 0

print(f"Loss = structure + {DICE_W}×Dice + {BOUNDARY_W}×Boundary")

# ── Train ─────────────────────────────────────────────────────────────────────

dataset    = FullDataset(TRAIN_IMAGE_PATH, TRAIN_MASK_PATH, 352, mode='train')
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8)

device = torch.device("cuda")
model  = SAM2UNet(HIERA_PATH).to(device)

optim     = opt.AdamW([{"params": model.parameters(), "initia_lr": LR}],
                      lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optim, EPOCH, eta_min=1e-7)

os.makedirs(SAVE_PATH, exist_ok=True)

for epoch in range(EPOCH):
    for i, batch in enumerate(dataloader):
        x      = batch['image'].to(device)
        target = batch['label'].to(device)

        optim.zero_grad()
        pred0, pred1, pred2 = model(x)

        loss = (combined_loss(pred0, target, DICE_W, BOUNDARY_W) +
                combined_loss(pred1, target, DICE_W, BOUNDARY_W) +
                combined_loss(pred2, target, DICE_W, BOUNDARY_W))
        loss.backward()
        optim.step()

        if i % 50 == 0:
            print(f"epoch:{epoch+1}-{i+1}: loss:{loss.item():.4f}")

    scheduler.step()

    if (epoch + 1) % 5 == 0 or (epoch + 1) == EPOCH:
        ckpt = os.path.join(SAVE_PATH, f'SAM2-UNet-{epoch+1}.pth')
        torch.save(model.state_dict(), ckpt)
        print(f'[Saved] {ckpt}')


Overwriting /content/SAM2-UNet/train_v2.py


In [ ]:
!ls -lh /content/ckpt/sam2unet_v2_smoke/

## 5. Full training (20 epochs)

In [ ]:
%cd /content/SAM2-UNet

!python train_v2.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/sam2unet_v2_dice_boundary/ \
  --epoch 20 \
  --batch_size 12 \
  --lr 0.001

/content/SAM2-UNet
Loss = structure + 0.3×Dice + 0×Boundary
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: loss:5.6554
epoch:1-51: loss:2.8765
epoch:1-101: loss:2.2197
epoch:2-1: loss:1.9237
epoch:2-51: loss:1.4216
epoch:2-101: loss:0.9542
epoch:3-1: loss:1.3034
epoch:3-51: loss:1.4900
epoch:3-101: loss:0.8054
epoch:4-1: loss:1.2088
epoch:4-51: loss:1.0355
epoch:4-101: loss:1.4019
epoch:5-1: loss:0.8901
epoch:5-51: loss:0.9228
epoch:5-101: loss:0.6746
[Saved] /content/ckpt/sam2unet_v2_dice_boundary/SAM2-UNet-5.pth
epoch:6-1: loss:1.0131
epoch:6-51: loss:0.8264
epoch:6-101: loss:0.5701
epoch:7-1: loss:0.6485
epoch:7-51: loss:0.9751
epoch:7-101: loss:0.8449
epoch:8-1: loss:0.7348
epoch:8-51: loss:0.9725
epoch:8-101: loss:1.4960
epoch:9-1: loss:0.5969
epoch:9-51: loss:0.5641
epoch:9-101: loss:0.5247
epoch:10-1: loss:0.5641
e

In [ ]:
!ls -lh /content/ckpt/sam2unet_v2_dice_boundary/

total 4.9G
-rw-r--r-- 1 root root 827M May 14 17:11 SAM2-UNet-10.pth
-rw-r--r-- 1 root root 827M May 14 17:17 SAM2-UNet-15.pth
-rw-r--r-- 1 root root 827M May 14 17:22 SAM2-UNet-20.pth
-rw-r--r-- 1 root root 827M May 14 16:44 SAM2-UNet-25.pth
-rw-r--r-- 1 root root 827M May 14 16:49 SAM2-UNet-30.pth
-rw-r--r-- 1 root root 827M May 14 17:05 SAM2-UNet-5.pth


## 6. Test — inference trên 5 datasets

In [ ]:
%%bash
cd /content/SAM2-UNet

for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo "============================="
  echo "Testing $DATASET"
  echo "============================="

  python test.py \
    --checkpoint /content/ckpt/sam2unet_v2_dice_boundary/SAM2-UNet-20.pth \
    --test_image_path /content/data/Polyp/TestDataset/$DATASET/images/ \
    --test_gt_path /content/data/Polyp/TestDataset/$DATASET/masks/ \
    --save_path /content/preds/sam2unet_v2/$DATASET/
done


Testing Kvasir
Saving cju0u82z3cuma0835wlxrnrjv.png
Saving cju15wdt3zla10801odjiw7sy.png
Saving cju16ach3m1da0993r1dq3sn2.png
Saving cju16whaj0e7n0855q7b6cjkm.png
Saving cju17z0qongpa0993de4boim4.png
Saving cju1amqw6p8pw0993d9gc5crl.png
Saving cju1bm8063nmh07996rsjjemq.png
Saving cju1c3218411b08014g9f6gig.png
Saving cju1cbokpuiw70988j4lq1fpi.png
Saving cju1cj3f0qi5n0993ut8f49rj.png
Saving cju1cqc7n4gpy0855jt246k68.png
Saving cju1ddr6p4k5z08780uuuzit2.png
Saving cju1f8w0t65en0799m9oacq0q.png
Saving cju1h89h6xbnx08352k2790o9.png
Saving cju1hp9i2xu8e0988u2dazk7m.png
Saving cju2hfqnmhisa0993gpleeldd.png
Saving cju2hjrqcvi2j0801bx1i6gxg.png
Saving cju2hos57llxm08359g92p6jj.png
Saving cju2hqt33lmra0988fr5ijv8j.png
Saving cju2lberzkdzm09938cl40pog.png
Saving cju2mh8t6p07008350e01tx2a.png
Saving cju2nnqrqzp580855z8mhzgd6.png
Saving cju2np2k9zi3v079992ypxqkn.png
Saving cju2omjpeqj5a0988pjdlb8l1.png
Saving cju2osuru0ki00855txo0n3uu.png
Saving cju2pag1f0s4r0878h52uq83s.png
Saving cju2rga4psq9n098

/content/SAM2-UNet/test.py:38: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test.py:38: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test.py:38: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test.py:38: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test.py:38: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corne

## 7. Evaluate — tính metrics

In [ ]:
%cd /content/SAM2-UNet

# Fix MSIoU init nếu chưa patch
!sed -i "s/MSIOU = py_sod_metrics.MSIoU()/MSIOU = py_sod_metrics.MSIoU(with_dynamic=True, with_adaptive=True, with_binary=True)/g" eval.py
print('eval.py patched')

/content/SAM2-UNet
eval.py patched


In [ ]:
%%bash
cd /content/SAM2-UNet

for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo "============================="
  echo "Evaluating $DATASET"
  echo "============================="

  python eval.py \
    --dataset_name $DATASET \
    --pred_path /content/preds/sam2unet_v2/$DATASET/ \
    --gt_path /content/data/Polyp/TestDataset/$DATASET/masks/
done


Evaluating Kvasir
[0] Processing cju0u82z3cuma0835wlxrnrjv.png...
[1] Processing cju15wdt3zla10801odjiw7sy.png...
[2] Processing cju16ach3m1da0993r1dq3sn2.png...
[3] Processing cju16whaj0e7n0855q7b6cjkm.png...
[4] Processing cju17z0qongpa0993de4boim4.png...
[5] Processing cju1amqw6p8pw0993d9gc5crl.png...
[6] Processing cju1bm8063nmh07996rsjjemq.png...
[7] Processing cju1c3218411b08014g9f6gig.png...
[8] Processing cju1cbokpuiw70988j4lq1fpi.png...
[9] Processing cju1cj3f0qi5n0993ut8f49rj.png...
[10] Processing cju1cqc7n4gpy0855jt246k68.png...
[11] Processing cju1ddr6p4k5z08780uuuzit2.png...
[12] Processing cju1f8w0t65en0799m9oacq0q.png...
[13] Processing cju1h89h6xbnx08352k2790o9.png...
[14] Processing cju1hp9i2xu8e0988u2dazk7m.png...
[15] Processing cju2hfqnmhisa0993gpleeldd.png...
[16] Processing cju2hjrqcvi2j0801bx1i6gxg.png...
[17] Processing cju2hos57llxm08359g92p6jj.png...
[18] Processing cju2hqt33lmra0988fr5ijv8j.png...
[19] Processing cju2lberzkdzm09938cl40pog.png...
[20] Process

/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 ins